In [ ]:
import os
import json
import base64
import logging
from openai import OpenAI
from dotenv import load_dotenv
from video_transcript import video_transcript
from object_detection import object_detection
from sentiment_analysis import sentiment_analysis
from question_answer import question_answer

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)

### Load API Key and Video

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
video_path = "data/AI_Intern_Project.mp4"

In [6]:
# model="gpt-4o-mini-transcribe"
# model="gpt-4o-transcribe"
# model="whisper-1"

transcription = video_transcript(client=client, video_path=video_path, model="whisper-1")
objects = object_detection(client=client, video_path=video_path, model="gpt-4.1", sample_rate=0.5)
mode_sentiment = sentiment_analysis(client=client, transcription=transcription, model="gpt-4.1")
qa_pairs = question_answer(client=client, transcription=transcription, model="gpt-4.1")

2025-11-08 06:21:19 [INFO] video_transcript: Extracting audio track...
2025-11-08 06:21:21 [INFO] video_transcript: Transcribing video...
2025-11-08 06:21:27 [INFO] object_detection: Sampling video...
2025-11-08 06:21:29 [INFO] object_detection: Detecting objects...
2025-11-08 06:22:50 [INFO] sentiment_analysis: Analysing mode and sentiment...
2025-11-08 06:22:54 [INFO] question_answer: Generating Q&A pairs...


In [7]:
# Merge and format output
merge_output = {
    "Transcription": transcription,
    "Objects": json.loads(objects).get("objects", []),
    "Mode and sentiment": json.loads(mode_sentiment),
    "Q&A pairs": json.loads(qa_pairs).get("QA_pairs", [])
}

json_output = json.dumps(merge_output, indent=4, ensure_ascii=False).replace(',\n    "', ',\n\n    "')

In [8]:
print(json_output)

{
    "Transcription": "Cooking the perfect steak is not always easy, but if you follow these tips I learned at culinary school, it'll be a whole lot easier. Today I'm using a beautiful thick cut New York strip. If you like it medium rare, the thicker the better. Next give it a generous coating of salt and pepper all over from high above. Grab a good pan, get it nice and hot, then add some neutral oil and add the steak fat cap down. Once the fat is rendered, rotate it onto its side. To get a beautiful even sear, apply some gentle even pressure. After a few minutes, take a peek and flip when it's a deep golden brown. Immediately add a knob of butter and some aromatics and begin basting the steak. To finish cooking it, transfer into a 375 degree oven for five minutes or so. In the meantime, use the pan drippings to make a quick red wine pan sauce by sauteing some shallots, deglazing with red wine, and once it's all set, add some stock and reduce by 75%. Finish it off with cold butter and

### Image Generation

In [44]:
# Build input content
dev_content = [
    {
        "type": "input_text",
        "text": "The image must clearly show all objects, without omitting or obscuring."
    }
]

usr_content = [
    {
        "type": "input_text",
        "text": f"Create an image that includes all the items in the list: {json.loads(objects).get("objects", [])}"
    }
]

In [45]:
response = client.responses.create(
    model="gpt-5",
    input=[
        {"role": "developer", "content": dev_content},
        {"role": "user", "content": usr_content}
    ],
    tools=[{"type": "image_generation"}],
)

In [ ]:
# Save the image to a file
image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

if image_data:
    with open("data/output.png", "wb") as f:
        f.write(base64.b64decode(image_data[0]))

### Anime Generation

In [ ]:
with open("data/OnePiece_Sanji.jpg", "rb") as image_file:
    anime_image =  base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
# Build input content
anime_usr_content = [
    {
        "type": "input_text",
        "text": (
            "Replace the person with the individual from the given image."
            "Replicate the artistic style and texture of the given image."
        )
    },
    {
        "type": "input_image",
        "image_url": f"data:image/jpeg;base64,{anime_image}"
    }
]

In [ ]:
anime_response = client.responses.create(
    model="gpt-5",
    previous_response_id=response.id,
    input=[
        {
            "role": "user",
            "content": anime_usr_content
        }
    ],
    tools=[{"type": "image_generation"}],
)

In [ ]:
# Save the image to a file
anime_data = [
    output.result
    for output in anime_response.output
    if output.type == "image_generation_call"
]

if anime_data:
    with open("data/anime_output.png", "wb") as f:
        f.write(base64.b64decode(anime_data[0]))